# Term streaming

A propagated `PauliTermSum`/`MajoranaTermSum` can grow very large before truncation
catches up, and even after truncation, a term sum from a real workload can be too
big to comfortably hold several copies of in memory at once. propaq's gzip term
streamer is built for that: `save`/`from_file` round-trip a term sum through a
compressed binary file, and `PauliTermStreamer` reads one back lazily, one term at a
time, instead of materializing the whole file into memory first. This notebook
propagates a moderately large observable, saves it, and compares the eager
(`from_file`) and streaming (`PauliTermStreamer` + `merge_from_file`) ways of
reading it back.

## Setup

A toy circuit is enough here: term streaming is a data-plumbing feature, not a
physics one, so we'll build a wider, deeper random circuit than the very first
notebook's to get a term sum with enough terms to make saving and streaming it
meaningful.

In [3]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import (
    XXPlusYYGate,
    PhaseGate,
    RZGate,
    CPhaseGate,
    SwapGate,
    XGate,
)

GATES = [
    (lambda: XXPlusYYGate(np.random.uniform(0, 2 * np.pi), np.random.uniform(0, 2 * np.pi)), 2),
    (lambda: PhaseGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: RZGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: CPhaseGate(np.random.uniform(0, 2 * np.pi)), 2),
    (lambda: SwapGate(), 2),
    (lambda: XGate(), 1),
]

np.random.seed(0)
n_qubits = 8
qc = QuantumCircuit(n_qubits)

for _ in range(120):
    factory, nq = GATES[np.random.randint(len(GATES))]
    gate = factory()
    qubits = np.random.choice(n_qubits, size=nq, replace=False).tolist()
    qc.append(gate, qubits)

from qiskit.quantum_info import SparsePauliOp

observable = SparsePauliOp.from_list([("X" + "I" * (n_qubits - 1), 1.0)])

## Propagating and writing to disk in one step

`PauliPropagator.propagate` returns the fully propagated `PauliTermSum` directly
(unlike `expectation_value`, which only returns the scalar result and per-gate
`n_terms`). It also accepts an optional `filename`, which writes the propagated
observable straight to a gzip-compressed file as part of propagation, so there's no
separate save step needed if you already know you want it on disk.

In [4]:
from propaq.circuits import PauliCircuit
from propaq.datatypes import PauliTermSum
from propaq.propagators import PauliPropagator
from propaq import FlushSchedule

obs_term_sum = PauliTermSum.from_sparse_pauli_op(observable)
pc = PauliCircuit.from_qiskit(qc)

prop = PauliPropagator(schedule=FlushSchedule(merge_max_terms=1), n_threads=4, progress_bar=True)
propagated = prop.propagate(obs_term_sum, pc, filename="propagated_terms.gz")

print("Number of terms:", len(propagated.items()))
print("norm_squared:   ", propagated.norm_squared())

Propagating through gates: 100%|██████████| 281/281 [00:00<00:00, 614.28it/s, terms=65535]


Number of terms: 65535
norm_squared:    0.9999999999999991


## Reading it back eagerly

`PauliTermSum.from_file` loads the whole file into a fresh `PauliTermSum` in one
call: simple, and fine as long as the result comfortably fits in memory.

In [5]:
reloaded = PauliTermSum.from_file("propagated_terms.gz")

print("Number of terms:", len(reloaded.items()))
print("norm_squared:   ", reloaded.norm_squared())
print("Exactly matches the original:", dict(reloaded.items()) == dict(propagated.items()))

Number of terms: 65535
norm_squared:    1.0000000000000013
Exactly matches the original: True


## Reading it back lazily

`PauliTermStreamer.from_file` opens the same file but yields one `(term, coefficient)`
pair at a time as you iterate, without ever holding the full contents in memory.
That's useful just to peek at a large file's contents, and it's also what
`merge_from_file` uses internally to fold a streamed file directly into an existing
`PauliTermSum` one term at a time, with no intermediate full-file allocation at all.

In [6]:
from propaq._rust_core import PauliTermStreamer

streamer = PauliTermStreamer.from_file("propagated_terms.gz")
print("First 5 terms in the file:")
for i, (term, coeff) in enumerate(streamer):
    if i >= 5:
        break
    print(f"  weight={term.weight:<2d} coeff={coeff}")

First 5 terms in the file:
  weight=8  coeff=2.369962251292547e-05
  weight=7  coeff=-0.0007067802853712364
  weight=5  coeff=-0.0017317286259556002
  weight=5  coeff=7.07054764506067e-05
  weight=7  coeff=-0.00012106000104901261


In [7]:
streamed_accumulator = PauliTermSum()
streamed_accumulator.merge_from_file(PauliTermStreamer.from_file("propagated_terms.gz"))

print("Number of terms:", len(streamed_accumulator.items()))
print("norm_squared:   ", streamed_accumulator.norm_squared())
print("Exactly matches the original:", dict(streamed_accumulator.items()) == dict(propagated.items()))

Number of terms: 65535
norm_squared:    1.0000000000000013
Exactly matches the original: True
